# Rodent vs. No-Rodent Classifier — EfficientNet-Lite0 (Colab)

Fine-tunes an ImageNet-pretrained EfficientNet-Lite0 on your rodent / no-rodent dataset,
then exports a quantized TFLite model for edge deployment.

**Before running:** set `RODENT_DIR` and `NO_RODENT_DIR` in the "Configure paths" cell below.


## 1. Setup

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # must be set BEFORE tensorflow is imported anywhere

!pip install -q tensorflow-hub tf_keras scikit-learn



In [2]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.metrics import classification_report, confusion_matrix

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Mount Drive and configure paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ---- EDIT THIS ----
DATASET_ROOT = "/content/ei_curated"   # folder containing training/ and testing/
# --------------------

TRAIN_DIR = os.path.join(DATASET_ROOT, "training")
TEST_DIR  = os.path.join(DATASET_ROOT, "testing")

OUTPUT_DIR = "/content/drive/MyDrive/rodent_model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224          # EfficientNet-Lite0 native input size
BATCH_SIZE = 32
SEED = 42


## 3. Build a file list from the Edge Impulse export

Each of `training/` and `testing/` is a flat folder of images plus one `bounding_boxes.labels`
JSON file (the Edge Impulse bounding-box label format). An image counts as `rodent` if it has
at least one bounding box listed; otherwise (missing from the file, or an empty box list --
which includes your generated background images) it's `no_rodent`.

No grouping/leakage handling here, per your note that the dataset is already cleaned and reduced.

In [5]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def parse_ei_labels(folder):
    """Reads bounding_boxes.labels from an Edge Impulse export folder and returns
    {filename: has_rodent_bool}."""
    labels_path = os.path.join(folder, "bounding_boxes.labels")
    with open(labels_path, "r") as f:
        data = json.load(f)

    boxes_by_file = data.get("boundingBoxes", data)  # tolerate slightly different EI export shapes
    has_rodent = {}
    for fname, boxes in boxes_by_file.items():
        has_rodent[fname] = bool(boxes) and len(boxes) > 0
    return has_rodent

def collect_files(folder):
    folder = Path(folder)
    has_rodent = parse_ei_labels(folder)

    rows = []
    for p in folder.iterdir():
        if p.suffix.lower() not in VALID_EXTS:
            continue
        # default to no_rodent if the file isn't mentioned in the labels JSON at all
        # (this is exactly the case for your generated background images)
        is_rodent = has_rodent.get(p.name, False)
        rows.append({
            "filepath": str(p),
            "label": "rodent" if is_rodent else "no_rodent",
        })
    return pd.DataFrame(rows)

df_train_full = collect_files(TRAIN_DIR)
df_test = collect_files(TEST_DIR)

df_train_full["label_idx"] = (df_train_full["label"] == "rodent").astype(int)
df_test["label_idx"] = (df_test["label"] == "rodent").astype(int)

print("Training folder:")
print(df_train_full["label"].value_counts())
print("\nTesting folder:")
print(df_test["label"].value_counts())


Training folder:
label
rodent       3600
no_rodent     480
Name: count, dtype: int64

Testing folder:
label
rodent       900
no_rodent    120
Name: count, dtype: int64


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Train / val split (testing/ is already your held-out test set)

Edge Impulse already separated testing/ for you, so that becomes the test set as-is. We just
split training/ into train and val -- a plain stratified random split, no grouping, per your
note that the dataset is already cleaned.

In [7]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    stratify=df_train_full["label_idx"],
    random_state=SEED,
)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Train: {len(df_train)} images")
print(f"Val:   {len(df_val)} images")
print(f"Test:  {len(df_test)} images")
print("\nClass balance (train):")
print(df_train['label'].value_counts(normalize=True))


Train: 3468 images
Val:   612 images
Test:  1020 images

Class balance (train):
label
rodent       0.882353
no_rodent    0.117647
Name: proportion, dtype: float64


## 5. Build tf.data pipelines

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0   # EfficientNet-Lite hub module expects [0,1]
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(df_split, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df_split["filepath"].values,
                                              df_split["label_idx"].values.astype(np.float32)))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1000, seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(df_train, training=True)
val_ds   = make_dataset(df_val,   training=False)
test_ds  = make_dataset(df_test,  training=False)


## 6. Build the model

Uses the ImageNet-pretrained EfficientNet-Lite0 feature vector from TensorFlow Hub, frozen
initially, with a small classification head on top.

In [9]:
EFFNET_LITE0_URL = "https://tfhub.dev/tensorflow/efficientnet/lite0/feature-vector/2"

def build_model(trainable_backbone=False):
    backbone = hub.KerasLayer(EFFNET_LITE0_URL, input_shape=(IMG_SIZE, IMG_SIZE, 3),
                               trainable=trainable_backbone)
    model = tf.keras.Sequential([
        backbone,
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    return model

model = build_model(trainable_backbone=False)
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 1280)              3413024   
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 1)                 1281      
                                                                 
Total params: 3414305 (13.02 MB)
Trainable params: 1281 (5.00 KB)
Non-trainable params: 3413024 (13.02 MB)
_________________________________________________________________


## 7. Phase 1 — train the classification head (backbone frozen)

In [10]:
# If your classes are imbalanced, weight them so the minority class isn't ignored
n_rodent = (df_train["label_idx"] == 1).sum()
n_no_rodent = (df_train["label_idx"] == 0).sum()
total = n_rodent + n_no_rodent
class_weight = {
    0: total / (2 * n_no_rodent),
    1: total / (2 * n_rodent),
}
print("Class weights:", class_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "phase1_best.keras"),
                                        monitor="val_loss", save_best_only=True),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks,
)


Class weights: {0: np.float64(4.25), 1: np.float64(0.5666666666666667)}
Epoch 1/15
109/109 [==============================] - 22s 121ms/step - loss: 0.4447 - accuracy: 0.8849 - precision: 0.9609 - recall: 0.9065 - val_loss: 0.2642 - val_accuracy: 0.9444 - val_precision: 0.9828 - val_recall: 0.9537
Epoch 2/15
109/109 [==============================] - 17s 134ms/step - loss: 0.2567 - accuracy: 0.9490 - precision: 0.9842 - recall: 0.9575 - val_loss: 0.2388 - val_accuracy: 0.9395 - val_precision: 0.9903 - val_recall: 0.9407
Epoch 3/15
109/109 [==============================] - 16s 110ms/step - loss: 0.1885 - accuracy: 0.9634 - precision: 0.9916 - recall: 0.9667 - val_loss: 0.1816 - val_accuracy: 0.9526 - val_precision: 0.9885 - val_recall: 0.9574
Epoch 4/15
109/109 [==============================] - 14s 95ms/step - loss: 0.1565 - accuracy: 0.9651 - precision: 0.9923 - recall: 0.9680 - val_loss: 0.1387 - val_accuracy: 0.9624 - val_precision: 0.9868 - val_recall: 0.9704
Epoch 5/15
109/109 [=

## 8. Phase 2 — fine-tune the backbone

Unfreeze the backbone and continue training with a much lower learning rate so we adapt
the pretrained features without destroying them.

In [11]:
model_ft = build_model(trainable_backbone=True)
model_ft.set_weights(model.get_weights())  # carry over phase-1 weights

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "phase2_best.keras"),
                                        monitor="val_loss", save_best_only=True),
]

history2 = model_ft.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks_ft,
)


ValueError: Exception encountered when calling layer "keras_layer_1" (type KerasLayer).

in user code:

    File "/usr/local/lib/python3.12/dist-packages/tensorflow_hub/keras_layer.py", line 226, in call  *
        self._check_trainability()
    File "/usr/local/lib/python3.12/dist-packages/tensorflow_hub/keras_layer.py", line 285, in _check_trainability  *
        raise ValueError(

    ValueError: Setting hub.KerasLayer.trainable = True is unsupported when loading from the TF1 Hub format.


Call arguments received by layer "keras_layer_1" (type KerasLayer):
  • inputs=tf.Tensor(shape=(None, 224, 224, 3), dtype=float32)
  • training=None

## 9. Evaluate on the held-out test set

In [14]:
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_ds)
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Test precision: {test_prec:.4f}")
print(f"Test recall:    {test_rec:.4f}")

y_true = df_test["label_idx"].values
y_pred_prob = model.predict(test_ds).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["no_rodent", "rodent"]))

print("Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:")
print(confusion_matrix(y_true, y_pred))


32/32 [==============================] - 6s 174ms/step - loss: 0.1109 - accuracy: 0.9539 - precision: 0.9874 - recall: 0.9600
Test accuracy:  0.9539
Test precision: 0.9874
Test recall:    0.9600
32/32 [==============================] - 3s 77ms/step

Classification report:
              precision    recall  f1-score   support

   no_rodent       0.75      0.91      0.82       120
      rodent       0.99      0.96      0.97       900

    accuracy                           0.95      1020
   macro avg       0.87      0.93      0.90      1020
weighted avg       0.96      0.95      0.96      1020

Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:
[[109  11]
 [ 36 864]]


## 10. Convert to TFLite and quantize

`dynamic_range` quantization is a safe default (smaller model, minimal accuracy loss, runs on
most edge devices). If your target device needs full int8 (e.g. a microcontroller), use the
`full_integer` cell instead, which needs a representative dataset sample.

In [16]:
saved_model_dir = os.path.join(OUTPUT_DIR, "saved_model")
model.export(saved_model_dir)

# --- Option A: dynamic-range quantization (good default) ---
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic = converter.convert()

with open(os.path.join(OUTPUT_DIR, "model_dynamic_range.tflite"), "wb") as f:
    f.write(tflite_dynamic)

print("Saved model_dynamic_range.tflite —",
      len(tflite_dynamic) / 1024, "KB")


Saved artifact at '/content/drive/MyDrive/rodent_model_output/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_layer_input')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132099869292496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769109648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769109840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769110608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769111184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769110032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769110224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769110416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769111376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132098769111760: TensorSpec(shape=(), dty

In [17]:
# --- Option B: full integer quantization (for microcontrollers / strict edge targets) ---
def representative_dataset():
    for images, _ in train_ds.take(50):   # ~50 batches is usually enough
        for img in images:
            yield [tf.expand_dims(img, 0)]

converter_int8 = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8

tflite_int8 = converter_int8.convert()

with open(os.path.join(OUTPUT_DIR, "model_int8.tflite"), "wb") as f:
    f.write(tflite_int8)

print("Saved model_int8.tflite —", len(tflite_int8) / 1024, "KB")


Saved model_int8.tflite — 3858.5546875 KB


## 11. Verify the quantized model still performs well

In [19]:
!pip install -q ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 19.2 MB/s eta 0:00:00


In [20]:
from ai_edge_litert.interpreter import Interpreter
# then use Interpreter(...) instead of tf.lite.Interpreter(...)

In [21]:
def evaluate_tflite(tflite_path, df_split, is_int8=False):
    interpreter = Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct = 0
    for _, row in df_split.iterrows():
        img = tf.io.read_file(row["filepath"])
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        if is_int8:
            img = tf.cast(img * 255, tf.uint8)
        img = tf.expand_dims(img, 0)

        interpreter.set_tensor(input_details["index"], img)
        interpreter.invoke()
        pred = interpreter.get_tensor(output_details["index"])[0][0]
        if is_int8:
            pred = pred / 255.0
        pred_label = 1 if pred > 0.5 else 0
        correct += int(pred_label == row["label_idx"])

    return correct / len(df_split)

acc_dynamic = evaluate_tflite(os.path.join(OUTPUT_DIR, "model_dynamic_range.tflite"), df_test, is_int8=False)
print(f"Dynamic-range TFLite test accuracy: {acc_dynamic:.4f}")

acc_int8 = evaluate_tflite(os.path.join(OUTPUT_DIR, "model_int8.tflite"), df_test, is_int8=True)
print(f"Int8 TFLite test accuracy: {acc_int8:.4f}")


Dynamic-range TFLite test accuracy: 0.9549
Int8 TFLite test accuracy: 0.9520


## 12. Save labels file (needed alongside the .tflite model for inference)

In [22]:
labels = ["no_rodent", "rodent"]
with open(os.path.join(OUTPUT_DIR, "labels.txt"), "w") as f:
    f.write("\n".join(labels))

print("All outputs saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))


All outputs saved to: /content/drive/MyDrive/rodent_model_output
['phase1_best.keras', 'saved_model', 'model_dynamic_range.tflite', 'model_int8.tflite', 'labels.txt']


## 13. Package the int8 model as an Arduino library

This converts `model_int8.tflite` into a C byte array and assembles a proper Arduino library
folder (`library.properties`, `src/`, an example sketch), then zips it — ready to install via
Arduino IDE's *Sketch → Include Library → Add .ZIP Library*.

**One real dependency you do need to install yourself (this notebook can't do it for you):**
the [TensorFlow Lite for Microcontrollers Arduino library](https://github.com/tensorflow/tflite-micro-arduino-examples)
(search "Arduino_TensorFlowLite" or "Chirale_TensorFlowLite" in the Arduino Library Manager).
Your generated library depends on it at compile time — it's not bundled here since it's a
separate, actively-maintained project outside this pipeline.

In [23]:
LIBRARY_NAME = "RodentDetector"   # change if you want a different library/class name

lib_root = os.path.join(OUTPUT_DIR, LIBRARY_NAME)
src_dir = os.path.join(lib_root, "src")
example_dir = os.path.join(lib_root, "examples", f"{LIBRARY_NAME}_Inference")

os.makedirs(src_dir, exist_ok=True)
os.makedirs(example_dir, exist_ok=True)


### 13a. Convert the .tflite file into a C byte array

In [24]:
def tflite_to_c_array(tflite_path, var_name="g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()

    lines = []
    for i in range(0, len(data), 12):
        chunk = data[i:i + 12]
        lines.append(", ".join(f"0x{b:02x}" for b in chunk))
    body = ",\n  ".join(lines)

    header = f"""// Auto-generated from {os.path.basename(tflite_path)} -- do not edit by hand.
#ifndef {var_name.upper()}_DATA_H
#define {var_name.upper()}_DATA_H

alignas(8) const unsigned char {var_name}[] = {{
  {body}
}};
const unsigned int {var_name}_len = {len(data)};

#endif  // {var_name.upper()}_DATA_H
"""
    return header, len(data)

model_header, model_size = tflite_to_c_array(
    os.path.join(OUTPUT_DIR, "model_int8.tflite"), var_name="g_model"
)

with open(os.path.join(src_dir, "model_data.h"), "w") as f:
    f.write(model_header)

print(f"Embedded model: {model_size} bytes ({model_size/1024:.1f} KB) -> src/model_data.h")


Embedded model: 3951160 bytes (3858.6 KB) -> src/model_data.h


### 13b. Write library.properties

In [25]:
library_properties = f"""name={LIBRARY_NAME}
version=1.0.0
author=Your Name
maintainer=Your Name <you@example.com>
sentence=On-device rodent / no-rodent classification using a quantized EfficientNet-Lite0 model.
paragraph=Runs TensorFlow Lite for Microcontrollers inference on a quantized int8 EfficientNet-Lite0 model to classify camera frames as rodent or no_rodent.
category=Sensors
url=
architectures=*
depends=Arduino_TensorFlowLite
"""

with open(os.path.join(lib_root, "library.properties"), "w") as f:
    f.write(library_properties)


### 13c. Write the library header/source

A thin wrapper class around the TFLite Micro interpreter. It expects a `{IMG_SIZE}x{IMG_SIZE}x3`
uint8 image buffer as input and returns a float probability (rodent likelihood) as output.

In [26]:
header_code = f"""#ifndef {LIBRARY_NAME.upper()}_H
#define {LIBRARY_NAME.upper()}_H

#include <TensorFlowLite.h>

class {LIBRARY_NAME} {{
  public:
    {LIBRARY_NAME}();
    bool begin();
    // input: pointer to {IMG_SIZE}*{IMG_SIZE}*3 uint8 RGB bytes, already resized to model input size
    // returns: probability that the image contains a rodent (0.0 - 1.0)
    float predict(const uint8_t* image_data);

  private:
    bool _initialized = false;
}};

#endif  // {LIBRARY_NAME.upper()}_H
"""

with open(os.path.join(src_dir, f"{LIBRARY_NAME}.h"), "w") as f:
    f.write(header_code)


In [27]:
cpp_code = f"""#include "{LIBRARY_NAME}.h"
#include "model_data.h"

#include <tensorflow/lite/micro/micro_interpreter.h>
#include <tensorflow/lite/micro/micro_mutable_op_resolver.h>
#include <tensorflow/lite/micro/micro_log.h>
#include <tensorflow/lite/schema/schema_generated.h>

namespace {{
  constexpr int kTensorArenaSize = 200 * 1024;  // tune down/up depending on your board's RAM
  alignas(16) uint8_t tensor_arena[kTensorArenaSize];

  const tflite::Model* model = nullptr;
  tflite::MicroInterpreter* interpreter = nullptr;
  TfLiteTensor* input = nullptr;
  TfLiteTensor* output = nullptr;
}}

{LIBRARY_NAME}::{LIBRARY_NAME}() {{}}

bool {LIBRARY_NAME}::begin() {{
  model = tflite::GetModel(g_model);
  if (model->version() != TFLITE_SCHEMA_VERSION) {{
    MicroPrintf("Model schema version mismatch!");
    return false;
  }}

  // Add only the ops your model actually needs (keeps flash usage down).
  // If you get an "unresolved op" error at runtime, add the missing op here.
  static tflite::MicroMutableOpResolver<8> resolver;
  resolver.AddConv2D();
  resolver.AddDepthwiseConv2D();
  resolver.AddFullyConnected();
  resolver.AddLogistic();
  resolver.AddAveragePool2D();
  resolver.AddReshape();
  resolver.AddAdd();
  resolver.AddMul();

  static tflite::MicroInterpreter static_interpreter(
      model, resolver, tensor_arena, kTensorArenaSize);
  interpreter = &static_interpreter;

  if (interpreter->AllocateTensors() != kTfLiteOk) {{
    MicroPrintf("AllocateTensors() failed");
    return false;
  }}

  input = interpreter->input(0);
  output = interpreter->output(0);
  _initialized = true;
  return true;
}}

float {LIBRARY_NAME}::predict(const uint8_t* image_data) {{
  if (!_initialized) return -1.0f;

  memcpy(input->data.uint8, image_data, input->bytes);

  if (interpreter->Invoke() != kTfLiteOk) {{
    MicroPrintf("Invoke() failed");
    return -1.0f;
  }}

  // Output is quantized uint8 -- dequantize back to a 0.0-1.0 probability
  uint8_t raw = output->data.uint8[0];
  float scale = output->params.scale;
  int zero_point = output->params.zero_point;
  return (raw - zero_point) * scale;
}}
"""

with open(os.path.join(src_dir, f"{LIBRARY_NAME}.cpp"), "w") as f:
    f.write(cpp_code)


### 13d. Write an example sketch

In [28]:
example_sketch = f"""#include <{LIBRARY_NAME}.h>

{LIBRARY_NAME} detector;

// Fill this with a real {IMG_SIZE}x{IMG_SIZE}x3 uint8 RGB frame from your camera before calling
// detector.predict(). Left as a placeholder here.
uint8_t image_buffer[{IMG_SIZE} * {IMG_SIZE} * 3];

void setup() {{
  Serial.begin(115200);
  while (!Serial) {{}}

  if (!detector.begin()) {{
    Serial.println("Failed to initialize model.");
    while (1) {{}}
  }}
  Serial.println("Model ready.");
}}

void loop() {{
  // TODO: capture a frame into image_buffer here, resized to {IMG_SIZE}x{IMG_SIZE}
  float rodent_prob = detector.predict(image_buffer);
  Serial.print("Rodent probability: ");
  Serial.println(rodent_prob);

  delay(1000);
}}
"""

with open(os.path.join(example_dir, f"{LIBRARY_NAME}_Inference.ino"), "w") as f:
    f.write(example_sketch)


### 13e. Zip the library

In [29]:
zip_path_base = os.path.join(OUTPUT_DIR, LIBRARY_NAME)
zip_path = shutil.make_archive(zip_path_base, "zip", root_dir=OUTPUT_DIR, base_dir=LIBRARY_NAME)

print("Arduino library zipped at:", zip_path)
print("\nInstall via Arduino IDE: Sketch -> Include Library -> Add .ZIP Library... -> select this file.")
print("Remember to also install the 'Arduino_TensorFlowLite' (or 'Chirale_TensorFlowLite') library")
print("from the Library Manager first -- this generated library depends on it.")


Arduino library zipped at: /content/drive/MyDrive/rodent_model_output/RodentDetector.zip

Install via Arduino IDE: Sketch -> Include Library -> Add .ZIP Library... -> select this file.
Remember to also install the 'Arduino_TensorFlowLite' (or 'Chirale_TensorFlowLite') library
from the Library Manager first -- this generated library depends on it.
